# Downsampling

**Learning goals:**
* Experiencing the effects of interpolation and decimation on the frequency content of a signal
* Understanding the role of filters in sample rate conversion

**Relevant reading:**
* Chapters 10.1 - 10.6 in the book "Understanding Digital Signal Processing"

**Libraries and notebook config:**

In [ ]:
from scipy.io import wavfile
import scipy.signal as sig
import numpy as np
from IPython.display import Audio
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [8.00, 4.5]
plt.rcParams["figure.autolayout"] = True
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.xmargin"] = 0.0

%matplotlib inline

### Problem description

Given a continuous-time signal $x(t)$ which has the following frequency content: 
$$X(\omega) = \begin{cases}1 - \frac{|f|}{40}, & -40\text{ Hz} < f < 40\text{ Hz} \\ 0 ,& \text{otherwise} \end{cases}$$

The signal $x(t)$ is sampled with a initial sampling rate $f_s = 100 \text{ Hz}$, to produce a discrete-time signal $x[n]$. The sampled signal $x[n]$ is recorded in the file `task1_signal.mat` in the `data` folder. The code cell below will load the signal $x[n]$ into an array `x_n` as well as the initial sampling frequency $f_s$ into the variable `fs`, and use the matlab-function `magnitude_spectrum` to create a plot of the signal's frequency content.

In [ ]:
from scipy.io import loadmat
data = loadmat("data/task1_signal.mat", squeeze_me=True)
x_n = data["x_n"]
fs = data["fs"]

plt.magnitude_spectrum(x_n, Fs=fs);
plt.title("Magnitude spectrum of signal stored in 'task1_signal.mat'")
plt.show()

Throughout this problem we will explore how sample rate conversion can be performed on the signal $x[n]$, and what steps need to be taken to ensure the frequency content after sample rate conversion matches that seen in the plot above to the largest possible degree.


### Downsampling
The first component of sample rate conversion we will explore is Downsampling; the process of reducing the sample rate of a discrete signal $x_{\text{old}}[n]$ by a factor of $M$. Described mathematically, the downsampled signal $x_{\text{new}}[m]$ is a discrete-time signal where the new sampling frequency is 
$$f_{s, \text{ new}} = \frac{f_{s, \text{ old}}}{M}$$
Before we simply reduce the number of samples in a signal however, it is important to first remove any frequency components which reside above the *new* nyquist frequency $\frac{f_{s, \text{ new}}}{2}$. Otherwise, the high-frequency portions of the signal $x[n]$ will cause aliasing in the downsampled signal $x_{\text{new}}[m]$. To illustrate why this is necessary, let's see what happens when we decimate the signal $x[n]$ without filtering first...

## a)
* Create a new array of downsampled data by using **list slicing** to decimate the signal `x_n` by a factor of $2$, and display the magnitude spectrum of the decimated signal alongside the original signal using the pyplot function `magnitude_spectrum`. Mathematically, the decimated signal will given by the following equation:

$$x_{\text{new}}[m] = x[2\cdot m]$$


In [ ]:
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task1a_lf.png) is an image showing what the end result should look like.

## b)
Study the magnitude spectrum of the decimated signal `x_new`. Can you explain why the frequency content in the decimated signal $x_{new}[m]$ has changed in the frequency range $10\text{Hz}<f<25\text{Hz}$?


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>


The unfortunate effect encountered in problem **a)** can be avoided by ensuring the Nyquist Criterion for the new sampling frequency $f_{s, \text{new}}$ is met **before** the signal is decimated. To achieve this, the we subject the signal $x_{\text{old}}[n]$ to a lowpass filter before decimation, as illustrated in the figure below.

![](figures/downsampling.png)

## c) 
* Given a decimation factor $M = 2$, what should be the cutoff frequency $\hat{\omega}_c$ for the lowpass filter used in to downsample the signal from the data file?



In [ ]:
w_c = "???" # Replace with correct value (unit: rad/sample)
# WRITE YOUR CODE IN THIS CELL:

In [ ]:
from hashlib import sha1
assert sha1(str(round(w_c, 5)).encode('utf-8')+b'78f94').hexdigest() == 'fe1a095c0b33a97a8838711eb0eaa8f433ce3a82', 'Wrong answer for w_c :('; print('Correct answer for w_c :)')

Now that we have observed the effects of directly decimating the signal $x[n]$, lets now insert a filter before the decimation, and see what effects this will have on the downsampled signal's frequency content.

## d)

* Use either the *window design method* or the function [remez](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.remez.html) to design a $257$-tap lowpass FIR filter with the transition region centered around $\hat{\omega}_c$ found in problem **c)**. Then, filter the full-rate signal $x[n]$ using your lowpass filter **before** decimation. Once again, plot the magnitude spectrum of the downsampled signal $x_{\text{new}}[m]$ alongside the original signal $x[n]$. <br>
*PS: Use convolution `mode="same"` to ensure output length equals input length, thereby ensuring consistent scaling of the magnitude spectrum.*

<!--*PS: Filtering the signal using convolution will result in a longer output than input, but with the same total signal energy. Matlab's `magnitude_spectrum` function divides by signal lenght $N$ to calculate the magnitde spectrum, meaning the filtered magnitude spectrum may appear weaker than the input magnitude spectrum. This can be corrected by scaling with $\frac{L_{\text{out}}}{L_{\text{in}}}$*.-->

In [ ]:
# WRITE YOUR CODE IN THIS CELL:

[Here](figures/task1b_lf.png) is an image showing what the end result should look like. PS: don't worry too much if the magnitudes don't match exaclty, this is merely a scaling issue. The important part is that the *shape* of the downsampled magnitude spectrum is correct.

## e)

* Study the magnitude spectrum plot for $x_{\text{new}}[m]$ and $x[n]$. Explain why our current result would be more desirable as a output of a downsampling operation thant the result in problem **a)**


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>


### Good to know:

The above process of downsampling is fairly standard, and specialized DSP libraries have functions to perform downsampling in both Python and C for STM32.
* **Python:** The function [`scipy.signal.decimate`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.decimate.html) may be used. Includes automatic filter design, but requires a specified filter order.
* **STM32:** The CMSIS-DSP has separte [FIR Decimator functions](https://arm-software.github.io/CMSIS-DSP/main/group__FIR__decimate.html). Requires user to design a FIR filter (in Python), and store filter coeffeicients directly in the C code as an array. These functions are optimized to make use of pylyphase decimation techniques, significantly reducing the number of multiplications needed at the filtering stage of the decimator.